# 03 EEA Batch Ingestion

## Purpose

Use EEA historical air-quality data as the required file/batch source. Normalize local EEA rows, map stations to `city_id`, run data-quality checks, aggregate to city/day/pollutant level, and write Silver Parquet.


## Inputs

- `data/silver/city_reference.parquet` from Phase 2.
- Local EEA CSV or Parquet files under `data/bronze/eea/`.
- If no local EEA file exists, the notebook creates a tiny controlled sample under `data/samples/` to keep the transformation reproducible.


## Outputs

- `data/silver/eea_city_daily.parquet`


## Technologies used

Python, pandas, pyarrow, Parquet, Jupyter Notebook.


## Configuration

No EEA download is performed here. Generated sample/output data is ignored by Git. Historical EEA data is not mixed with Open-Meteo API data.


In [1]:
from pathlib import Path
import os
import json
import pandas as pd

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))
CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"


## Implementation

The implementation follows the course reference pattern: load file data with pandas, clean/normalize columns, write Parquet, and validate read-back. It is scoped only to PM2.5, PM10 and NO2.


In [2]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd

BRONZE_EEA_DIR = DATA_DIR / "bronze" / "eea"
SAMPLES_DIR = DATA_DIR / "samples"
SILVER_DIR = DATA_DIR / "silver"
for path in [BRONZE_EEA_DIR, SAMPLES_DIR, SILVER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CORE_POLLUTANTS = {"pm2_5", "pm10", "no2"}
POLLUTANT_LABEL_MAP = {
    "PM2.5": "pm2_5",
    "PM2,5": "pm2_5",
    "Particles < 2.5 µm (aerodynamic diameter)": "pm2_5",
    "PM10": "pm10",
    "Particles < 10 µm (aerodynamic diameter)": "pm10",
    "NO2": "no2",
    "Nitrogen dioxide": "no2",
    "Nitrogen dioxide (air)": "no2",
}

STATION_MAPPING = pd.DataFrame([
    {"city_id": "vienna_at", "eea_station_id": "AT90TAB", "mapping_status": "selected"},
    {"city_id": "berlin_de", "eea_station_id": "DEBE068", "mapping_status": "selected"},
    {"city_id": "paris_fr", "eea_station_id": "FR04143", "mapping_status": "selected"},
    {"city_id": "madrid_es", "eea_station_id": "ES0118A", "mapping_status": "selected"},
    {"city_id": "rome_it", "eea_station_id": "IT1906A", "mapping_status": "selected"},
    {"city_id": "amsterdam_nl", "eea_station_id": "NL00014", "mapping_status": "selected"},
    {"city_id": "warsaw_pl", "eea_station_id": "PL0592A", "mapping_status": "selected"},
    {"city_id": "prague_cz", "eea_station_id": "CZ0ARIE", "mapping_status": "selected"},
])

city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
assert set(STATION_MAPPING["city_id"]).issubset(set(city_reference_df["city_id"])), \
    f"STATION_MAPPING city_ids not in city reference: {set(STATION_MAPPING['city_id']) - set(city_reference_df['city_id'])}"

def create_controlled_eea_sample(path: Path) -> Path:
    base_date = datetime(2023, 1, 1, tzinfo=timezone.utc)
    rows = []
    for day_offset in range(30):
        date = base_date + timedelta(days=day_offset)
        for hour in range(0, 24, 8):
            ts = (date + timedelta(hours=hour)).strftime("%Y-%m-%dT%H:%M:%SZ")
            for station_index, station in enumerate(STATION_MAPPING["eea_station_id"]):
                variation = (day_offset % 7) + (hour // 8)
                rows.extend([
                    {"AirQualityStationEoICode": station, "DatetimeBegin": ts, "AirPollutant": "PM2.5", "Concentration": round(8.0 + variation + station_index * 0.5, 1), "Unit": "µg/m³"},
                    {"AirQualityStationEoICode": station, "DatetimeBegin": ts, "AirPollutant": "PM10", "Concentration": round(18.0 + variation + station_index * 0.8, 1), "Unit": "µg/m³"},
                    {"AirQualityStationEoICode": station, "DatetimeBegin": ts, "AirPollutant": "Nitrogen dioxide", "Concentration": round(30.0 + variation + station_index * 1.2, 1), "Unit": "µg/m³"},
                ])
    pd.DataFrame(rows).to_csv(path, index=False)
    return path

def first_existing(df: pd.DataFrame, candidates: list[str]) -> str:
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(f"Missing expected column. Tried: {candidates}")

def load_eea_raw(path: Path) -> pd.DataFrame:
    raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    station_col = first_existing(raw, ["AirQualityStationEoICode", "AirQualityStation", "station_id"])
    ts_col = first_existing(raw, ["DatetimeBegin", "datetime_begin", "timestamp", "date"])
    pollutant_col = first_existing(raw, ["AirPollutant", "pollutant", "Pollutant", "Component"])
    value_col = first_existing(raw, ["Concentration", "concentration", "Value", "value"])
    unit_col = first_existing(raw, ["Unit", "unit"])
    df = pd.DataFrame({
        "eea_station_id": raw[station_col].astype(str).str.strip(),
        "datetime_begin": pd.to_datetime(raw[ts_col], utc=True, errors="coerce"),
        "pollutant": raw[pollutant_col].astype(str).str.strip().map(POLLUTANT_LABEL_MAP),
        "value": pd.to_numeric(raw[value_col], errors="coerce"),
        "unit": raw[unit_col].astype(str).str.strip(),
    })
    df = df.dropna(subset=["eea_station_id", "datetime_begin", "pollutant", "value", "unit"])
    df = df[df["value"] >= 0].copy()
    df = df[df["pollutant"].isin(CORE_POLLUTANTS)].copy()
    return df.reset_index(drop=True)

def map_stations_to_cities(raw_df: pd.DataFrame) -> pd.DataFrame:
    selected = STATION_MAPPING.query("mapping_status == 'selected'")[["eea_station_id", "city_id"]]
    return raw_df.merge(selected, on="eea_station_id", how="inner")

def aggregate_to_city_daily(mapped_df: pd.DataFrame) -> pd.DataFrame:
    required = ["city_id", "datetime_begin", "pollutant", "value", "unit"]
    missing = [col for col in required if col not in mapped_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    df = mapped_df.copy()
    df["datetime_begin"] = pd.to_datetime(df["datetime_begin"], utc=True, errors="coerce")
    if df[required].isna().any().any():
        raise ValueError("Required EEA fields contain nulls")
    df = df[df["value"].ge(0) & df["pollutant"].isin(CORE_POLLUTANTS)].copy()
    df["date"] = df["datetime_begin"].dt.date
    daily = df.groupby(["city_id", "date", "pollutant", "unit"], as_index=False).agg(
        mean_value=("value", "mean"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        observation_count=("value", "count"),
    )
    daily["source"] = "eea"
    daily["processing_time_utc"] = pd.Timestamp(datetime.now(timezone.utc))
    return daily


In [3]:
source_files = sorted(BRONZE_EEA_DIR.glob("*.csv")) + sorted(BRONZE_EEA_DIR.glob("*.parquet"))
if source_files:
    eea_input_path = source_files[0]
else:
    eea_input_path = create_controlled_eea_sample(SAMPLES_DIR / "eea_controlled_sample.csv")

raw_eea = load_eea_raw(eea_input_path)
mapped_eea = map_stations_to_cities(raw_eea)
eea_city_daily = aggregate_to_city_daily(mapped_eea)
output_path = SILVER_DIR / "eea_city_daily.parquet"
eea_city_daily.to_parquet(output_path, index=False)
eea_city_daily.head()


,city_id,date,pollutant,unit,mean_value,min_value,max_value,observation_count,source,processing_time_utc
0,amsterdam_nl,2023-01-01,no2,µg/m³,37.0,36.0,38.0,3,eea,2026-05-31 13:40:33.865803+00:00
1,amsterdam_nl,2023-01-01,pm10,µg/m³,23.0,22.0,24.0,3,eea,2026-05-31 13:40:33.865803+00:00
2,amsterdam_nl,2023-01-01,pm2_5,µg/m³,11.5,10.5,12.5,3,eea,2026-05-31 13:40:33.865803+00:00
3,amsterdam_nl,2023-01-02,no2,µg/m³,38.0,37.0,39.0,3,eea,2026-05-31 13:40:33.865803+00:00
4,amsterdam_nl,2023-01-02,pm10,µg/m³,24.0,23.0,25.0,3,eea,2026-05-31 13:40:33.865803+00:00


## Validation / Quality Checks

Validate pollutant scope, required schema, mapping coverage, non-negative values, aggregation counts and Parquet read-back.


In [4]:
required_output_columns = {
    "city_id", "date", "pollutant", "mean_value", "min_value", "max_value",
    "observation_count", "unit", "source", "processing_time_utc",
}
assert required_output_columns.issubset(eea_city_daily.columns)
assert set(eea_city_daily["pollutant"]).issubset(CORE_POLLUTANTS)
assert (eea_city_daily["observation_count"] >= 1).all()
assert (eea_city_daily["source"] == "eea").all()
roundtrip = pd.read_parquet(output_path)
assert len(roundtrip) == len(eea_city_daily)
roundtrip.groupby("pollutant")["observation_count"].sum()


pollutant
no2      720
pm10     720
pm2_5    720
Name: observation_count, dtype: int64

## Results

Phase 3 produces `eea_city_daily.parquet` as the historical Silver air-quality dataset. If no real EEA extract is present, the output is based on a clearly marked controlled sample and must be replaced with real EEA data for final analysis.


## Limitations

Station-to-city mapping is simplified and must be reviewed against real EEA station metadata. The controlled sample is only a reproducibility fallback, not analytical evidence.


## Next step

Run notebook `04_wikipedia_web_scraping.ipynb` to build contextual city metadata from Wikipedia.
